# Trabajo Final — Pipeline de pedidos de TiendaBSG
**Tópicos Avanzados para Data Science en Python**

**Alumno(a):** _escribe aquí tu nombre_
**Repositorio público:** _pega aquí el link de tu repositorio en GitHub_

---

Eres el *data engineer* de **TiendaBSG**, una tienda online con despachos en 6 ciudades del Perú. Cada día el sistema de ventas deja un archivo CSV con los pedidos del día en la carpeta `datos/raw/`. Tu trabajo es procesar los pedidos de **octubre 2026** de principio a fin:

| Parte | Qué harás | Capítulo | Puntos |
|---|---|---|---|
| 1. Revisar los archivos | Explorar `raw/`, separar archivos vacíos y contar registros | 1. Línea de comandos | 3.5 |
| 2. Analizar con Spark | Limpiar, filtrar, agrupar y contar con DataFrames o Spark SQL | 3. Spark | 12 |
| 3. Cerrar el proceso | Mover los CSV procesados y dejar el resultado final listo | 1. Línea de comandos | 2 |
| 4. Entregar | Subir este notebook y el resultado a tu repositorio público | 2. Git | 2.5 |
| | | **Total** | **20** |

**Reglas**
- Ejecuta las celdas **en orden**. Si algo se rompe, vuelve a ejecutar la **Preparación** (reinicia los datos) y continúa.
- En la Parte 2 usa solo **DataFrames** (API de PySpark o Spark SQL). **No uses RDDs.**
- Responde las preguntas como **comentarios** (`# ...`) en tu propia celda de código.
- Antes de descargar el notebook: **Entorno de ejecución → Reiniciar y ejecutar todo**, para que se vean todas las salidas en GitHub.

## Preparación (no se califica)
Ejecuta estas tres celdas. Instalan PySpark, crean la sesión de Spark y generan los archivos del mes:

```
/content/trabajo_final/
├── datos/
│   ├── raw/          ← 31 archivos pedidos_2026-10-DD.csv (uno por día)
│   ├── procesados/   ← vacía: aquí moverás los CSV ya procesados (Parte 3)
│   └── cuarentena/   ← vacía: aquí moverás los archivos con problemas (Parte 1)
└── resultados/       ← vacía: aquí guardarás el resumen final (Partes 2 y 3)
```

In [ ]:
# Java + PySpark (tarda ~1 minuto la primera vez)
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q uninstall -y dataproc-spark-connect  # paquete de Colab que pide PySpark 4; no lo usamos
!pip -q install "pyspark==3.5.*"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("BSG-TrabajoFinal")
         .config("spark.ui.showConsoleProgress", "false")   # sin barras de progreso
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                     # oculta los logs de la JVM
print("Versión de Spark:", spark.version)

In [ ]:
# Genera los pedidos de octubre 2026 (volver a ejecutar = reiniciar los datos)
import csv, os, random
from datetime import date

TF = "/content/trabajo_final"
RAW = f"{TF}/datos/raw"
PROCESADOS = f"{TF}/datos/procesados"
CUARENTENA = f"{TF}/datos/cuarentena"
RESULTADOS = f"{TF}/resultados"

shutil.rmtree(TF, ignore_errors=True)
for carpeta in (RAW, PROCESADOS, CUARENTENA, RESULTADOS):
    os.makedirs(carpeta)

rng = random.Random(2026)
CIUDADES = {"Lima": 40, "Arequipa": 15, "Trujillo": 12, "Chiclayo": 10, "Cusco": 13, "Piura": 10}
CATALOGO = {  # producto: (categoria, precio de lista, peso en las ventas)
    "Laptop": ("Tecnologia", 3200.0, 8), "Celular": ("Tecnologia", 1500.0, 15),
    "Audifonos": ("Tecnologia", 180.0, 20), "Licuadora": ("Hogar", 250.0, 12),
    "Cafetera": ("Hogar", 320.0, 8), "Zapatillas": ("Moda", 280.0, 22),
    "Casaca": ("Moda", 220.0, 15)}
DEVOLUCION = {"Tecnologia": 4, "Hogar": 3, "Moda": 12}  # % de pedidos devueltos
PAGOS = {"Tarjeta": 40, "Yape": 30, "Efectivo": 15, "Transferencia": 15}
DIAS_VACIOS = {12, 25}  # el sistema falló y dejó estos archivos vacíos

id_pedido, dia_anterior = 0, []
for dia in range(1, 32):
    fecha = date(2026, 10, dia).isoformat()
    ruta = f"{RAW}/pedidos_{fecha}.csv"
    if dia in DIAS_VACIOS:
        open(ruta, "w").close()
        continue
    filas = []
    for _ in range(rng.randint(1500, 2500)):
        id_pedido += 1
        ciudad = rng.choices(list(CIUDADES), weights=CIUDADES.values())[0]
        producto = rng.choices(list(CATALOGO), weights=[v[2] for v in CATALOGO.values()])[0]
        categoria, precio_lista, _ = CATALOGO[producto]
        precio = round(precio_lista * rng.uniform(0.9, 1.1), 2)
        cantidad = rng.randint(1, 4)
        devolucion = DEVOLUCION[categoria]
        estado = rng.choices(["ENTREGADO", "CANCELADO", "DEVUELTO"],
                             weights=[90 - devolucion, 10, devolucion])[0]
        pago = rng.choices(list(PAGOS), weights=PAGOS.values())[0]
        # Errores típicos del sistema de origen
        r = rng.random()
        if r < 0.10:
            ciudad = ciudad.upper()
        elif r < 0.20:
            ciudad = f" {ciudad.lower()}  "
        if rng.random() < 0.05:
            estado = estado.lower()
        if rng.random() < 0.02:
            cantidad = ""
        if rng.random() < 0.01:
            precio = -precio
        filas.append([id_pedido, fecha, ciudad, categoria, producto, cantidad, precio, pago, estado])
    duplicados = rng.sample(dia_anterior, len(dia_anterior) // 50)  # pedidos reenviados
    dia_anterior = filas
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["id_pedido", "fecha", "ciudad", "categoria", "producto",
                    "cantidad", "precio_unitario", "metodo_pago", "estado"])
        w.writerows(filas + duplicados)

%cd {TF}
!ls datos/raw | head -3
print("...")

## Parte 1 — Revisar los archivos (Línea de comandos · 3.5 pts)
Antes de procesar nada, un *data engineer* revisa qué llegó. Usa **solo comandos de terminal** (`!comando` para una línea, o `%%bash` al inicio de la celda para varias). Ya estás ubicado en `/content/trabajo_final`.

| Comando | ¿Qué hace? |
|---|---|
| `ls -lh carpeta` | Lista archivos con su tamaño |
| `ls carpeta \| wc -l` | Cuenta cuántos archivos hay |
| `du -sh carpeta` | Tamaño total de una carpeta |
| `head -n 5 archivo` | Muestra las primeras líneas |
| `find carpeta -type f -empty` | Busca archivos vacíos |
| `mv origen destino` | Mueve (o renombra) archivos |
| `wc -l archivos` | Cuenta líneas |
| `grep -ci texto` | Cuenta líneas que contienen un texto (sin importar mayúsculas) |

### ✏️ Ejercicio 1.1 — Explorar la carpeta raw (1 pts)
1. Lista los archivos de `datos/raw` con su tamaño.
2. Cuenta cuántos archivos hay y el tamaño total de la carpeta.
3. Muestra las 5 primeras líneas del archivo del 1 de octubre.
4. Responde: ¿qué columnas tiene cada archivo? ¿Ves algún archivo raro en el listado?

In [ ]:
%%bash
# Tu código aquí

### ✏️ Ejercicio 1.2 — Separar los archivos vacíos (1.5 pts)
1. Encuentra los archivos vacíos de `datos/raw` con `find`.
2. Muévelos a `datos/cuarentena` (con `mv` o con `find ... -exec mv {} destino \;`).
3. Comprueba: lista `datos/cuarentena` y cuenta cuántos archivos quedan en `datos/raw`.
4. Responde: ¿por qué conviene apartarlos en lugar de borrarlos?

In [ ]:
%%bash
# Tu código aquí

### ✏️ Ejercicio 1.3 — Contar registros antes de Spark (1 pts)
1. Cuenta las líneas de todos los CSV de `datos/raw` juntos con `wc -l` (la última línea muestra el total).
2. Calcula cuántas **filas de datos** hay: total de líneas menos una cabecera por archivo. Anota el número: lo compararás en el Ejercicio 2.1.
3. Con `grep`, cuenta cuántas líneas contienen la palabra `cancelado` sin importar mayúsculas.

In [ ]:
%%bash
# Tu código aquí

## Parte 2 — Analizar con Spark (DataFrames · 12 pts)
Cada ejercicio puedes resolverlo con la **API de DataFrames** o con **Spark SQL**; el Ejercicio 2.7 es obligatoriamente con SQL. **No uses RDDs.**

| API de DataFrames | Spark SQL | ¿Qué hace? |
|---|---|---|
| `df.select("a", "b")` | `SELECT a, b` | Elegir columnas |
| `df.filter(F.col("a") > 0)` | `WHERE a > 0` | Filtrar filas |
| `df.dropDuplicates(["id"])` | `SELECT DISTINCT ...` | Quitar duplicados |
| `df.dropna(subset=["a"])` | `WHERE a IS NOT NULL` | Quitar nulos |
| `df.withColumn("t", expr)` | `SELECT ..., expr AS t` | Crear o reemplazar una columna |
| `df.groupBy("a").agg(F.sum("b"))` | `GROUP BY a` | Agrupar y agregar |
| `df.groupBy("a").count()` | `COUNT(*)` | Contar por grupo |
| `df.orderBy(F.desc("b"))` | `ORDER BY b DESC` | Ordenar |
| `df.createOrReplaceTempView("v")` | `spark.sql("SELECT ... FROM v")` | Usar SQL sobre un DataFrame |

> 💡 Para que las sumas grandes no salgan en notación científica: `.cast("decimal(18,2)")` o `CAST(SUM(x) AS DECIMAL(18,2))`.

### ✏️ Ejercicio 2.1 — Leer los pedidos (1 pts)
1. Lee **todos** los CSV de `datos/raw` en un solo DataFrame llamado `pedidos` (pista: `spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)`).
2. Muestra el esquema con `printSchema()` y cuenta las filas.
3. Responde: ¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.2 — Explorar la calidad de los datos (1.5 pts)
1. Muestra 5 filas solo de las columnas `fecha`, `ciudad`, `producto`, `cantidad` y `estado`.
2. Muestra los valores distintos de `ciudad` y cuéntalos. ¿Por qué hay más de 6?
3. Cuenta cuántas filas tienen `cantidad` nula y cuántas tienen `precio_unitario` menor o igual a 0.
4. Cuenta cuántos `id_pedido` están repetidos (filas totales menos `id_pedido` distintos).

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.3 — Limpiar los pedidos (2 pts)
Crea el DataFrame `df_limpio` a partir de `pedidos`:
1. Elimina los duplicados por `id_pedido`.
2. Elimina las filas con `cantidad` nula y con `precio_unitario <= 0`.
3. Normaliza `ciudad` (quita espacios y deja la primera letra en mayúscula: `F.initcap(F.trim(...))`).
4. Normaliza `estado` en mayúsculas y sin espacios (`F.upper(F.trim(...))`).
5. Agrega la columna `total = cantidad * precio_unitario` redondeada a 2 decimales.

Muestra cuántas filas había antes y después, y los valores distintos de `ciudad` (deben ser 6) y de `estado` (deben ser 3).

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.4 — Filtrar pedidos (1.5 pts)
1. Crea el DataFrame `entregados` solo con los pedidos en estado `ENTREGADO` y cuéntalos.
2. ¿Cuántos pedidos entregados en **Lima**, pagados con **Yape**, tienen un `total` mayor a **1000**?
3. Muestra los 5 pedidos entregados de mayor `total` (columnas `id_pedido`, `ciudad`, `producto`, `cantidad`, `total`).

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.5 — Conteos (1.5 pts)
1. Cuenta los pedidos por `estado`, de mayor a menor.
2. Cuenta los pedidos por `metodo_pago`, de mayor a menor.
3. Responde: ¿qué porcentaje de los pedidos se cancela? ¿Cuál es el método de pago más usado?

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.6 — Ventas por ciudad (2 pts)
Con los pedidos **entregados**, crea el DataFrame `ventas_por_ciudad` con una fila por ciudad y estas columnas:
- `pedidos`: número de pedidos
- `unidades`: suma de `cantidad`
- `ventas`: suma de `total` (en decimal, sin notación científica)
- `ticket_promedio`: promedio de `total` redondeado a 2 decimales

Ordénalo de mayor a menor `ventas` y muéstralo. Responde: ¿qué ciudad vende más y cuál menos?

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.7 — Las mismas preguntas con Spark SQL (1.5 pts)
**Este ejercicio es obligatorio con SQL.**
1. Registra `df_limpio` como la vista temporal `pedidos_limpios`.
2. Con SQL, obtén por `categoria` el número de pedidos **entregados** y la suma de sus ventas, de mayor a menor.
3. Con SQL, obtén por `categoria` el número de pedidos **devueltos**, de mayor a menor. Responde: ¿qué categoría tiene más devoluciones?

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 2.8 — Guardar el resultado (1 pts)
1. Guarda `ventas_por_ciudad` como CSV con cabecera en `resultados/ventas_por_ciudad_spark`, en **un solo archivo** (pista: `coalesce(1)` y `mode("overwrite")`).
2. Lista la carpeta que creó Spark. Responde: ¿por qué Spark crea una **carpeta** y no un archivo?

In [ ]:
# Tu código aquí

## Parte 3 — Cerrar el proceso (Línea de comandos · 2 pts)
Spark ya terminó. Ahora deja las carpetas ordenadas para el próximo mes: lo que está en `raw/` ya se procesó y el resultado debe tener un nombre claro.

### ✏️ Ejercicio 3.1 — Mover los CSV procesados (1 pts)
1. Mueve todos los CSV de `datos/raw` a `datos/procesados` con `mv`.
2. Comprueba que `datos/raw` quedó vacía y cuenta cuántos archivos hay en `datos/procesados`.
3. Responde: ¿por qué es importante sacar los archivos de `raw/` después de procesarlos?

In [ ]:
%%bash
# Tu código aquí

### ✏️ Ejercicio 3.2 — Dejar el resultado con un nombre claro (1 pts)
1. Mueve el archivo `part-*.csv` que creó Spark a `resultados/ventas_por_ciudad.csv` (con `mv` también se renombra).
2. Borra la carpeta `resultados/ventas_por_ciudad_spark`.
3. Muestra el contenido de `resultados/ventas_por_ciudad.csv` y la estructura final con `find datos resultados -type f | sort`.

In [ ]:
%%bash
# Tu código aquí

## Parte 4 — Entregar con Git (2.5 pts)
Esta parte se hace **en la terminal de tu computadora** (Git Bash en Windows, Terminal en Mac/Linux), siguiendo los pasos del documento del trabajo final.

1. Descarga el resultado ejecutando la celda de abajo (se guarda en tu carpeta de Descargas).
2. Escribe tu nombre y el link de tu repositorio al inicio de este notebook.
3. **Entorno de ejecución → Reiniciar y ejecutar todo**, y luego **Archivo → Descargar → Descargar .ipynb**.
4. Sube los dos archivos a tu repositorio público con `git add`, `git commit` y `git push`.

In [ ]:
from google.colab import files
files.download(f"{RESULTADOS}/ventas_por_ciudad.csv")